# Phase D v4 — bounded M6 Colab execution
Run cells in order. A fresh Run all stops at an executable real-data gate before CIFAR-10 download or training, and a second executable gate stops before the full pilot. Markdown is explanatory only; CLI authorization flags are independently required. The official CIFAR-10 test partition is not used.

In [ ]:
from pathlib import Path
import hashlib, json, os, platform, subprocess, sys
import numpy as np, torch, torchvision
ROOT = Path('/content/probabilistic backpropagation')
assert ROOT.is_dir(), 'Upload or mount the repository at ROOT before continuing'
os.chdir(ROOT)
def run_checked(arguments):
    """Run one repository command and stop this cell on any nonzero exit."""
    return subprocess.run([sys.executable, '-B', *map(str, arguments)], check=True)
print(sys.version)
print('torch', torch.__version__, 'torchvision', torchvision.__version__, 'numpy', np.__version__)
assert torch.cuda.is_available(), 'CUDA is required; CPU fallback is forbidden'
gpu = torch.cuda.get_device_name(0)
print('GPU:', gpu, 'CUDA:', torch.version.cuda, 'cuDNN:', torch.backends.cudnn.version())
assert 'T4' in gpu, f'Expected an NVIDIA T4 before timing comparisons, got {gpu}'
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=False)
assert not torch.backends.cuda.matmul.allow_tf32 and not torch.backends.cudnn.allow_tf32


In [ ]:
# Optional Drive persistence after an explicit mount; active training stays local.
# from google.colab import drive
# drive.mount('/content/drive')
PERSISTENCE_ROOT = None  # e.g. Path('/content/drive/MyDrive/phase_d_v4_pilot_seed0')
def persistence_arguments(run_id):
    """Return optional verified checkpoint-copy arguments for one run."""
    return [] if PERSISTENCE_ROOT is None else ['--persistence-dir', PERSISTENCE_ROOT/run_id]


In [ ]:
reference_manifest = json.loads(Path('configs/phase_d_v4_reference_manifest.json').read_text())
for name, record in reference_manifest['governing_artifacts'].items():
    digest = hashlib.sha256(Path(name).read_bytes()).hexdigest()
    assert digest == record['sha256'], (name, digest, record['sha256'])
for name in ['gradient_phase_d_resnet_v4.py', 'phase_d_v4_io.py', 'phase_d_v4_diagnostics.py', 'phase_d_v4_analysis.py', 'analyze_phase_d_v4.py', 'configs/phase_d_v4_reference.json', 'configs/phase_d_v4_pilot_seed0_manifest.json']:
    print(name, hashlib.sha256(Path(name).read_bytes()).hexdigest())
print('Protected identities verified')


In [ ]:
run_checked(['-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_phase_d_v4*.py', '-v'])
run_checked(['gradient_phase_d_resnet_v4.py', 'audit', '--device', 'cuda'])


In [ ]:
SMOKE_RESUMED = Path('/content/phase_d_v4_smoke_resumed')
SMOKE_FULL = Path('/content/phase_d_v4_smoke_uninterrupted')
assert not SMOKE_RESUMED.exists() and not SMOKE_FULL.exists(), 'Use fresh smoke directories'
run_checked(['gradient_phase_d_resnet_v4.py', 'smoke', '--device', 'cuda', '--out', SMOKE_RESUMED, '--stop-after-global-epoch', '0'])
partial = json.loads((SMOKE_RESUMED/'summary.json').read_text())
assert partial['completed'] is False and partial['checkpoint_sequence'] == 1
run_checked(['gradient_phase_d_resnet_v4.py', 'resume', '--device', 'cuda', '--out', SMOKE_RESUMED])
run_checked(['gradient_phase_d_resnet_v4.py', 'smoke', '--device', 'cuda', '--out', SMOKE_FULL])
run_checked(['analyze_phase_d_v4.py', 'compare-resume', SMOKE_FULL, SMOKE_RESUMED])
resumed_manifest = json.loads((SMOKE_RESUMED/'manifest.json').read_text())
resumed_summary = json.loads((SMOKE_RESUMED/'summary.json').read_text())
assert resumed_manifest['resumed'] is True and resumed_summary['completed'] is True
assert resumed_summary['checkpoint_sequence'] == 2
print('Synthetic resumed state equals uninterrupted state; timing and accuracy are not research evidence')


In [ ]:
AUTHORIZE_REAL_DATA_PREFLIGHT = False
if not AUTHORIZE_REAL_DATA_PREFLIGHT:
    raise RuntimeError('STOP: real-data preflight is disabled by default; obtain approval, set this value True, and rerun this cell')
# This approval covers dataset setup, two four-epoch warm-ups (eight full training-split passes), and bounded profiling only.
STUDY = Path('/content/runs_phase_d_v4/phase_d_v4_pilot_seed0')
CONFIGS = STUDY/'configs'
DATA = Path('/content/data')
SPLIT = STUDY/'split_indices.npz'
assert not STUDY.exists(), 'Use an explicit fresh study directory'
CONFIGS.mkdir(parents=True)
run_checked(['gradient_phase_d_resnet_v4.py', 'prepare-data', '--authorize-real-data', '--config', 'configs/phase_d_v4_reference.json', '--data-dir', DATA, '--split-file', SPLIT, '--download'])


In [ ]:
DONOR_ID = 'seed0_relative_feedback_donor'
DONOR_CONFIG = CONFIGS/f'{DONOR_ID}.json'
DONOR_RUN = STUDY/DONOR_ID
run_checked(['gradient_phase_d_resnet_v4.py', 'make-config', '--method', 'relative_feedback', '--run-id', DONOR_ID, '--output-root', STUDY, '--out-config', DONOR_CONFIG])
# Stop after global epoch 3: four warm-up epochs only, before calibration/post-warm training.
run_checked(['gradient_phase_d_resnet_v4.py', 'run', '--authorize-real-data', '--device', 'cuda', '--config', DONOR_CONFIG, '--data-dir', DATA, '--split-file', SPLIT, '--out', DONOR_RUN, '--stop-after-global-epoch', '3', *persistence_arguments(DONOR_ID)])
WARMUP_CKPT = DONOR_RUN/'checkpoints/checkpoint_000004.pt'
assert WARMUP_CKPT.is_file()
PLAIN_ID = 'seed0_exact_plain_preflight'
PLAIN_CONFIG = CONFIGS/f'{PLAIN_ID}.json'
PLAIN_RUN = STUDY/PLAIN_ID
run_checked(['gradient_phase_d_resnet_v4.py', 'make-config', '--method', 'exact', '--run-id', PLAIN_ID, '--output-root', STUDY, '--out-config', PLAIN_CONFIG])
run_checked(['gradient_phase_d_resnet_v4.py', 'run', '--authorize-real-data', '--device', 'cuda', '--config', PLAIN_CONFIG, '--data-dir', DATA, '--split-file', SPLIT, '--out', PLAIN_RUN, '--stop-after-global-epoch', '3', *persistence_arguments(PLAIN_ID)])
assert (PLAIN_RUN/'checkpoints/checkpoint_000004.pt').is_file()
print('Preflight warm-ups completed: 2 x 4 epochs = 8 full training-split passes')


In [ ]:
PREFLIGHT = STUDY/'resource_preflight'
run_checked(['gradient_phase_d_resnet_v4.py', 'preflight', '--authorize-real-data', '--device', 'cuda', '--config', DONOR_CONFIG, '--data-dir', DATA, '--split-file', SPLIT, '--checkpoint', WARMUP_CKPT, '--out', PREFLIGHT])
preflight = json.loads((PREFLIGHT/'profile_summary.json').read_text())
assert 'T4' in preflight['environment']['device_name']
required_components = {'plain_exact','exact_plus_predictor','learned_synthetic','zero_synthetic_shadow','skip_synthetic_shadow','controller_overhead'}
assert required_components.issubset(preflight['components'])
print(json.dumps(preflight['components'], indent=2))


In [ ]:
pilot = json.loads(Path('configs/phase_d_v4_pilot_seed0_manifest.json').read_text())
methods = [item['method'] for item in pilot['measured_conditions']]
expected_methods = ['exact','exact_standard_bn','relative_feedback','scheduled','relative_replay_zero_shadow','relative_replay_skip_shadow','relative_replay_exact_trunk','relative_periodic_epoch','relative_gap_permutation_epoch','relative_periodic_total','fixed_frontloaded']
assert methods == expected_methods
assert pilot['trajectory_accounting']['actual_full_trajectory_count'] == 13
assert len(pilot['run_order']['measured_block']) == 11
assert set(pilot['run_order']['measured_block']) == {item['run_id'] for item in pilot['measured_conditions']}
assert pilot['dependency_phase'][1]['depends_on'] == [DONOR_ID]
components = preflight['components']
slot_medians = [components[name]['median'] for name in ['plain_exact','exact_plus_predictor','learned_synthetic','zero_synthetic_shadow','skip_synthetic_shadow']]
slots = pilot['scientific_scope']['postwarm_epochs'] * pilot['scientific_scope']['training_batches_per_epoch']
trajectories = pilot['trajectory_accounting']['actual_full_trajectory_count']
predictor_warmup = json.loads((DONOR_RUN/'timing.json').read_text())['totals']['workflow_seconds']
plain_warmup = json.loads((PLAIN_RUN/'timing.json').read_text())['totals']['workflow_seconds']
actual_independent_warmups = pilot['trajectory_accounting']['plain_warmup_full_trajectory_attribution_count']*plain_warmup + pilot['trajectory_accounting']['predictor_warmup_full_trajectory_attribution_count']*predictor_warmup
scenario_low = actual_independent_warmups + trajectories * slots * min(slot_medians) / 1000
scenario_high = actual_independent_warmups + trajectories * slots * max(slot_medians) / 1000
checkpoint_bytes = preflight['checkpoint_size_bytes']
checkpoint_count = trajectories*pilot['scientific_scope']['total_epochs'] + pilot['scientific_scope']['warmup_epochs']
checkpoint_core_bytes = checkpoint_count*checkpoint_bytes
estimate = {
 'status':'preflight_scenarios_not_runtime_bounds',
 'actual_study_gpu_hours_scenario_low':scenario_low/3600,
 'actual_study_gpu_hours_scenario_high':scenario_high/3600,
 'standalone_method_costs':'reported separately from actual study resource use',
 'preflight_work_already_performed':'two four-epoch warm-ups = eight full training-split passes, plus bounded profile',
 'checkpoint_core_storage_gib_approx':checkpoint_core_bytes/2**30,
 'storage_assumption':f'current runner keeps all {checkpoint_count} epoch checkpoints; measured warm-up checkpoint size is a proxy',
 'storage_not_included':['resolved configs','split artifacts','masks','CSV/JSON logs','final predictors','analysis plots','checkpoint growth','Drive filesystem overhead'],
 'runtime_not_included_or_uncertain':['dataset setup','validation','calibration','checkpoint/log/Drive IO','finalization','hardware drift','thermal effects','mask-dependent slot mix'],
 'not_guaranteed_bounds':True, 'counts_are_not_flops':True}
print(json.dumps(pilot, indent=2))
print(json.dumps(estimate, indent=2))


## Full-pilot approval gate — stop here
Share the exact manifest, T4 record, verified resume smoke, component medians, scenario estimates, storage assumptions, and omissions. These are estimates, not guaranteed runtime bounds. Real-data preflight approval does not authorize donor completion or any full trajectory. Five seeds, optional ablations, retries, and sweeps remain unauthorized.

In [ ]:
AUTHORIZE_FULL_PILOT = False
if not AUTHORIZE_FULL_PILOT:
    raise RuntimeError('STOP: full pilot is disabled by default; obtain approval, set this value True, and rerun this cell')
run_checked(['gradient_phase_d_resnet_v4.py', 'resume', '--authorize-real-data', '--authorize-full-pilot', '--device', 'cuda', '--config', DONOR_CONFIG, '--data-dir', DATA, '--split-file', SPLIT, '--out', DONOR_RUN, *persistence_arguments(DONOR_ID)])
DONOR_MASK = DONOR_RUN/'masks.npz'
DONOR_MANIFEST = DONOR_RUN/'mask_manifest.json'
assert json.loads((DONOR_RUN/'summary.json').read_text())['completed'] is True


In [ ]:
# Resolve configs; this cell does not train.
conditions = [
 ('exact','seed0_exact_frozen',None,None), ('exact_standard_bn','seed0_exact_standard_bn',None,None),
 ('relative_feedback','seed0_relative_feedback_measured',None,None), ('scheduled','seed0_scheduled',None,None),
 ('relative_replay_learned','seed0_relative_replay_learned_equivalence',None,None),
 ('relative_replay_zero_shadow','seed0_relative_replay_zero_shadow',None,None),
 ('relative_replay_skip_shadow','seed0_relative_replay_skip_shadow',None,None),
 ('relative_replay_exact_trunk','seed0_relative_replay_exact_trunk',None,None),
 ('relative_periodic_epoch','seed0_relative_periodic_epoch',None,None),
 ('relative_gap_permutation_epoch','seed0_relative_gap_permutation_epoch_r0',None,0),
 ('relative_periodic_total','seed0_relative_periodic_total',None,None),
 ('fixed_frontloaded','seed0_fixed_frontloaded_learned','learned',None),
]
source_methods = {name for name, *_ in conditions if name.startswith('relative_replay_') or name in {'relative_periodic_epoch','relative_gap_permutation_epoch','relative_periodic_total'}}
for method, run_id, behavior, replica in conditions:
    arguments = ['gradient_phase_d_resnet_v4.py','make-config','--method',method,'--run-id',run_id,'--output-root',STUDY,'--out-config',CONFIGS/f'{run_id}.json']
    if behavior is not None: arguments += ['--synthetic-behavior', behavior]
    if replica is not None: arguments += ['--mask-replicate', str(replica)]
    if method in source_methods: arguments += ['--donor-mask', DONOR_MASK, '--donor-manifest', DONOR_MANIFEST]
    run_checked(arguments)
REPLAY_ID = 'seed0_relative_replay_learned_equivalence'
REPLAY_RUN = STUDY/REPLAY_ID
run_checked(['gradient_phase_d_resnet_v4.py','run','--authorize-real-data','--authorize-full-pilot','--device','cuda','--config',CONFIGS/f'{REPLAY_ID}.json','--data-dir',DATA,'--split-file',SPLIT,'--out',REPLAY_RUN,'--mask',DONOR_MASK,'--mask-manifest',DONOR_MANIFEST,*persistence_arguments(REPLAY_ID)])
run_checked(['analyze_phase_d_v4.py','compare-replay',DONOR_RUN,REPLAY_RUN])
PRE_BLOCK = STUDY/'pre_block_profile'
run_checked(['gradient_phase_d_resnet_v4.py','preflight','--authorize-real-data','--device','cuda','--config',DONOR_CONFIG,'--data-dir',DATA,'--split-file',SPLIT,'--checkpoint',WARMUP_CKPT,'--out',PRE_BLOCK])


In [ ]:
# Rerun this cell once per trajectory; it derives the next saved-order run.
ordered_ids = pilot['run_order']['measured_block']
completed_ids = {path.parent.name for path in STUDY.glob('*/summary.json') if json.loads(path.read_text()).get('completed')}
remaining = [run_id for run_id in ordered_ids if run_id not in completed_ids]
assert remaining, 'Measured block is complete; proceed to analysis and the post-block profile'
RUN_ID = remaining[0]
entry = next(item for item in pilot['measured_conditions'] if item['run_id'] == RUN_ID)
arguments = ['gradient_phase_d_resnet_v4.py','run','--authorize-real-data','--authorize-full-pilot','--device','cuda','--config',CONFIGS/f'{RUN_ID}.json','--data-dir',DATA,'--split-file',SPLIT,'--out',STUDY/RUN_ID,*persistence_arguments(RUN_ID)]
if entry['depends_on'] and entry['method'] != 'relative_feedback': arguments += ['--mask',DONOR_MASK,'--mask-manifest',DONOR_MANIFEST]
print('Executing saved-order trajectory:', RUN_ID)
run_checked(arguments)
if RUN_ID == 'seed0_relative_feedback_measured':
    run_checked(['analyze_phase_d_v4.py','compare-feedback-masks',DONOR_RUN,STUDY/RUN_ID])


In [ ]:
# Recovery: resume one partial run from its latest committed epoch boundary.
PARTIAL_RUN_ID = 'replace_with_partial_run_id'
# Use run_checked with resume, both authorization flags, the matching config/data/split, optional mask pair, and persistence_arguments.
# Restore only marker-verified copies. Resumed timing remains labeled non-clean and interruption/redo is retained.


In [ ]:
# Read-only validation and the required post-block drift profile after all saved-order runs finish.
completed = [str(path.parent) for path in STUDY.glob('*/summary.json') if json.loads(path.read_text()).get('completed')]
MEASURED = [path for path in completed if Path(path).name in set(pilot['run_order']['measured_block'])]
assert len(MEASURED) == len(pilot['run_order']['measured_block'])
run_checked(['analyze_phase_d_v4.py','validate',*MEASURED])
POST_BLOCK = STUDY/'post_block_profile'
run_checked(['gradient_phase_d_resnet_v4.py','preflight','--authorize-real-data','--device','cuda','--config',DONOR_CONFIG,'--data-dir',DATA,'--split-file',SPLIT,'--checkpoint',WARMUP_CKPT,'--out',POST_BLOCK])
import csv
from analyze_phase_d_v4 import hardware_drift_check
with (PRE_BLOCK/'profile_samples.csv').open() as handle: pre_samples = [float(row['latency_ms']) for row in csv.DictReader(handle) if row['component'] == 'plain_exact']
with (POST_BLOCK/'profile_samples.csv').open() as handle: post_samples = [float(row['latency_ms']) for row in csv.DictReader(handle) if row['component'] == 'plain_exact']
drift = hardware_drift_check(pre_samples, post_samples)
print(json.dumps(drift, indent=2))
assert not drift['flagged'], 'Hardware drift exceeded 10%; retain all records and seek authorization before repeating the entire block'
# Use a fresh analysis directory; one seed produces insufficient-sample intervals by design.
# run_checked(['analyze_phase_d_v4.py','summarize','--out',STUDY/'analysis',*MEASURED])
